# 🗄️ 4. 엔터프라이즈 Text2SQL: 시멘틱 레이어, 스키마 링킹, 값 연결

앞선 `3_Vector_and_Graph_RAG.ipynb`에서는 사내 규정이나 아키텍처 백서 같은 **비정형 문서**를 표준 Markdown 청크와 지식 그래프로 인덱싱하는 방법을 다뤘습니다.  
하지만 기업의 실시간 예산 집행액, 결재 대기 전표, 임직원 인사 정보, GPU 클러스터 가동률 같은 **정형 수치 데이터**는 관계형 데이터베이스(RDB / ERP)에 저장되어 있습니다.

표 데이터를 텍스트 청크로 잘라 벡터 DB에 넣으면 `top-k` 조각만 검색되므로 **전체 합계(`SUM`)나 비율 계산이 틀리고**, 결재 상태가 바뀔 때마다 재인덱싱해야 하는 문제가 발생합니다. 따라서 정형 DB는 **질문을 SQL로 변환해 DB 엔진이 직접 정확히 집계하도록 맡기는 Text2SQL** 방식으로 연결해야 합니다.

이번 노트북에서는 원시 `CREATE TABLE` 스키마만 LLM에 넘겼을 때 왜 **'조용한 오답(Silent Wrong Answer)'** 이 발생하는지 직접 확인하고, 이를 해결하는 엔터프라이즈 Text2SQL의 3대 핵심 기둥인 **시멘틱 레이어(Semantic Layer)**, **스키마 링킹(Schema Linking)**, **값 연결(Value Linking)** 과 **Self-Healing 실행 파이프라인(`rag/sql/`)** 을 단계별로 해부하고 조립합니다.

---

### 🎯 학습 목표
1. 정형 DB에 원시 DDL(`CREATE TABLE`)만 제공할 때 발생하는 **값 불일치(0행 반환)** 와 **도메인 모호성(조용한 오답)** 실패 원인을 재현하고 진단합니다.
2. DB에서 자동 추출한 물리 구조(`PRAGMA`) 위에 사람의 업무 지식(설명·규칙·지표 공식)을 입혀 **테이블·지표 카드('Text2SQL의 청크')** 로 만드는 **시멘틱 레이어(`DatabaseProfile`)** 를 이해합니다.
3. 수십 개 테이블 중 필요한 카드만 골라 상위 외래키(FK) 조인 경로를 보강하는 **스키마 링킹(`SchemaRetriever`)** 과, 일상 표현·약어를 DB 실제 셀 값에 매칭하는 **값 연결(`ValueLinker`)** 의 동작 원리와 상호 보완 구조를 익힙니다.
4. **읽기 전용 2중 가드레일(`executor.py`)** 과 **오류·0행 자가 수정(Self-Healing) 및 되묻기(`needs_clarification`) 워크플로우(`Text2SQL`)** 를 결합하고, 비정형 사내 규정 검색 결과(`context`)와 연동한 하이브리드 감사를 수행합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 모듈 / 기법 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트 탐색, `.env`, LLM 및 임베딩 모델 초기화 |
| **Part 1** | 원시 DDL 기반 Naive Text2SQL의 한계와 3대 해법 | `enterprise_erp.db` 구조 확인, 조용한 오답 체험, 3대 핵심 개념 한눈에 보기 |
| **Part 2** | 시멘틱 레이어 (Semantic Layer) | `DatabaseProfile` · 4계층 지식 결합(`PRAGMA` + `semantic_layer.yaml`) · 스키마 변경 감지 |
| **Part 3** | 스키마 링킹 (Schema Linking) | `SchemaRetriever` · 테이블/지표 카드 하이브리드 검색 + 상위(Parent) 방향 FK 확장 |
| **Part 4** | 값 연결 (Value Linking) | `ValueLinker` · 선택적 값 색인, 최장 일치 매칭(`'정수석'` vs `'수석'`), 누락 테이블 역보강 |
| **Part 5** | 2중 가드레일과 Self-Healing 워크플로우 | `validate_sql` · `run_readonly_query` · `Text2SQL` LangGraph 자가 수정 및 되묻기 분기 |
| **Part 6** | 비정형 규정 × 정형 ERP 에이전트 도구 결합 | `search_company_policy` + `search_enterprise_erp(context=...)` 여비 규정 위반 전표 감사 |
| **Part 7** | 정리 및 다음 단계 | Text2SQL 도구 5대 설계 원칙 요약 및 `5_MCP_and_Skills.ipynb` 연결 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`), LLM(`gemini-3.8-flash`), 임베딩 모델(`text-embedding-3-large`)을 초기화합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)
print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. LLM 및 임베딩 모델 초기화
from langchain_openai import OpenAIEmbeddings
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(f"✅ LLM 초기화 완료: {llm.model}")

---
## 💥 Part 1. 원시 DDL만 주면 왜 실패할까? (Text2SQL의 난제와 3대 해법)

### 1.1 실습 대상 데이터베이스: (주)넥스트AI 사내 ERP (`enterprise_erp.db`)

실습에 사용할 `app/database/enterprise_erp.db`는 현업 ERP처럼 **7개의 정규화된 테이블**로 구성되어 있습니다.  
단순 `SELECT *` 한 줄로는 복합 질문에 답할 수 없도록 외래키(FK)로 연결되어 있으며, 특히 `projects` 테이블에는 **실집행액 컬럼이 아예 없어** 반드시 지출 전표(`expense_transactions`)와 `JOIN`해 집계해야 합니다.

```mermaid
erDiagram
    departments ||--o{ employees : "소속 (dept_code)"
    employees ||--o{ employees : "직속상관 (manager_emp_id)"
    departments ||--o{ projects : "주관부서 (lead_dept_code)"
    employees ||--o{ projects : "총괄PM / 승인권자"
    projects ||--o{ project_members : "투입프로젝트 (project_id)"
    employees ||--o{ project_members : "투입인력 (emp_id)"
    projects ||--o{ expense_transactions : "예산집행 (project_id)"
    employees ||--o{ expense_transactions : "기안자 / 결재자"
    projects ||--o{ gpu_cluster_allocations : "할당클러스터 (project_id)"
    departments ||--o{ gpu_cluster_allocations : "관리부서 (managed_by_dept_code)"
    departments ||--o{ department_budgets : "비목별 운영예산 (dept_code)"
```

먼저 DB 안에 어떤 테이블과 몇 건의 행이 들어 있는지, 그리고 DB 자체에서 뽑아낸 **원시 스키마(`CREATE TABLE` DDL)** 가 어떻게 생겼는지 직접 확인해 보겠습니다.

In [ ]:
from rag.sql.executor import connect_readonly

ERP_DB_PATH = "app/database/enterprise_erp.db"
SEMANTIC_LAYER_PATH = "data/enterprise_erp_sql/semantic_layer.yaml"

conn = connect_readonly(ERP_DB_PATH)
ddl_rows = conn.execute(
    "SELECT name, sql FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY rowid"
).fetchall()

print("📊 [사내 ERP 테이블 현황]")
for table_name, _ in ddl_rows:
    cnt = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"  - {table_name:<26} : {cnt:>4}행")

ddl_map = dict(ddl_rows)
conn.close()

print("\n" + "=" * 75)
print("📄 [DB가 기본으로 아는 정보: departments & projects 원시 DDL]")
print(ddl_map["departments"])
print("-" * 75)
print(ddl_map["projects"])

### 1.2 실험: 원시 DDL(`CREATE TABLE`)만 프롬프트에 넣은 Naive Text2SQL의 실패

가장 단순한 Text2SQL 구현은 위 `CREATE TABLE` 문장들을 프롬프트에 그대로 붙여 넣고 LLM에게 SQL을 짜 달라고 요청하는 것입니다.  
하지만 실제 현업 질문을 던져 보면 **문법 오류조차 내지 않고 그럴듯한 표나 0행을 돌려주는 '조용한 오답(Silent Wrong Answer)'** 이 발생합니다. 두 가지 대표 사례를 직접 실행해 확인해 봅시다.

1. **질문 1 (일상 표현 vs DB 저장 값 불일치)**: `"클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘."`
   - 사용자는 흔히 `'클라우드팀'`이라고 부르지만, 실제 DB의 `departments`에는 정식 명칭 `'클라우드운영팀'`과 약칭 `'클운팀'`만 저장되어 있습니다.
2. **질문 2 (업무 규칙·지표 모호성)**: `"예산 집행률이 80% 이상인 항목을 모두 알려줘."`
   - 이 회사에는 **전사 전략 프로젝트 예산(`projects` + `expense_transactions`)** 과 **부서별 연간 운영 예산(`department_budgets`)** 두 가지 예산 원장이 공존합니다. 원시 DDL만 받은 모델은 어떻게 행동할까요?

In [ ]:
from pydantic import BaseModel, Field
from rag.sql import run_readonly_query

class RawSQLOutput(BaseModel):
    sql: str = Field(description="실행할 단일 SQLite SELECT 문")

naive_sql_llm = llm.with_structured_output(RawSQLOutput)
all_raw_ddls = "\n\n".join(sql for _, sql in ddl_rows)

def run_naive_text2sql(question: str) -> None:
    prompt = (
        "다음 SQLite CREATE TABLE 스키마만 보고 질문에 답하는 단일 SELECT SQL을 작성하세요.\n\n"
        f"{all_raw_ddls}\n\n질문: {question}"
    )
    draft = naive_sql_llm.invoke(prompt)
    res = run_readonly_query(ERP_DB_PATH, draft.sql)
    print(f"❓ 질문: {question}")
    print(f"📝 생성된 SQL:\n  {draft.sql}")
    print(f"📊 실행 결과 ({res.summary()}):\n{res.to_markdown()}\n" + "-" * 75)

run_naive_text2sql("클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘.")
run_naive_text2sql("예산 집행률이 80% 이상인 항목을 모두 알려줘.")

### 1.3 진단과 해법: 엔터프라이즈 Text2SQL을 지탱하는 3대 핵심 개념

위 두 실험 모두 **SQL 문법 오류는 단 한 건도 발생하지 않았습니다.** 그래서 더 위험합니다!
- **첫 번째 실패 (`0행` 반환)**: 모델이 `WHERE dept_name = '클라우드팀'`이라고 썼고, DB는 조용히 `(결과 0행)`을 반환했습니다. 사용자는 *"클라우드팀에는 편성된 예산이 없구나"* 라고 오해하게 됩니다.
- **두 번째 실패 (절반짜리 조용한 오답)**: `projects` 테이블에는 `spent_krw` 컬럼이 없고 `department_budgets`에만 `spent_krw`가 보이자, 모델은 사용자에게 묻지도 않고 **부서 운영 예산 한쪽만 조회**해 버렸습니다. 정작 집행률이 86.0%(`P-02`), 85.0%(`P-05`)로 경보 상태인 전략 프로젝트들은 통째로 누락되었습니다.

Text2SQL의 진짜 난제는 SQL 문법이 아니라 **"사람이 쓰는 업무 언어를 DB의 테이블 · 계산식 · 셀 값에 정확히 연결하는 것"** 입니다. 이를 해결하는 세 가지 핵심 개념을 쉬운 비유와 함께 정리하면 다음과 같습니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    Q["사용자 자연어 질문"] --> S["2. 스키마 링킹<br/>(Schema Linking)<br/>필요한 테이블·지표 카드 선별"]
    Q --> V["3. 값 연결<br/>(Value Linking)<br/>일상 표현 → DB 실제 셀 값"]
    SL["1. 시멘틱 레이어<br/>(Semantic Layer)<br/>DB 물리 구조 + 업무 뜻·공식"] -.->|"카드 색인 제공"| S
    SL -.->|"허용값·대상 컬럼 제공"| V
    S -->|"FROM · JOIN 재료"| G["SQL 생성 & 실행<br/>(2중 가드레일 + Self-Healing)"]
    V -->|"WHERE 조건 값 재료"| G

    style SL fill:#fff8e1,stroke:#f57f17,stroke-width:1.5px
    style S fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style V fill:#fff3e0,stroke:#fb8c00,stroke-width:1.5px
    style G fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

| 핵심 개념 | 직관적 비유 | SQL에서 맡는 역할 | 문서 RAG(`3_Vector_and_Graph_RAG`)와의 대응 | 구현 모듈 (`rag/sql/`) |
|:---|:---|:---|:---|:---|
| **1. 시멘틱 레이어**<br/>*(Semantic Layer)* | **신입 분석가용 업무 인수인계서**<br/>*(이름만 있는 장부에 뜻과 공식을 적음)* | **`SELECT` 계산식 · 해석 규칙**<br/>(`'실집행'`=승인완료만, 2종 예산 구분 등) | 표준 Markdown 정규화 & 문맥 헤더 | `DatabaseProfile`<br/>(`semantic_layer.py`) |
| **2. 스키마 링킹**<br/>*(Schema Linking)* | **수백 권 장부 중 필요한 장부만 꺼내기**<br/>*(연결된 상위 장부까지 세트로 챙김)* | **`FROM` · `JOIN` 대상 테이블 선별**<br/>(테이블·지표 카드 검색 + 상위 FK 확장) | `HybridRetriever` + `ContextualWindow` | `SchemaRetriever`<br/>(`schema_retriever.py`) |
| **3. 값 연결**<br/>*(Value Linking)* | **사내 은어 → DB 저장 코드 번역기**<br/>*(실제 셀에 적힌 글자 그대로 매칭)* | **`WHERE` 조건의 정확한 리터럴 값**<br/>(`'클운팀'`→`'클라우드운영팀'`, `'정수석'`≠`'수석'`) | `DomainDictionary` 약어 사전의 확장 | `ValueLinker`<br/>(`value_linker.py`) |

이제 Part 2부터 Part 4까지 세 기둥을 하나씩 직접 실행하며 내부 구조를 파헤쳐 봅시다.

---
## 📖 Part 2. 시멘틱 레이어 (Semantic Layer): 이름만 있는 DB에 '업무의 뜻' 붙이기

### 시멘틱 레이어란 무엇인가요?

데이터베이스는 `tx.amount_krw`가 숫자라는 것은 알지만, 우리 회사에서 **"실집행액"** 이라고 부를 때 `'승인완료'`만 더해야 하는지 `'결재대기'`도 더해야 하는지는 알지 못합니다.  
**시멘틱 레이어(Semantic Layer)** 는 BI 도구(dbt Semantic Layer, LookML 등)와 엔터프라이즈 Text2SQL에서 공통으로 쓰는 핵심 계층으로, **DB의 물리적 컬럼·테이블 위에 현업의 비즈니스 정의·동의어·계산 공식을 매핑한 'DB용 업무 사전'** 입니다.

우리 파이프라인(`rag/sql/semantic_layer.py`)은 유지보수 부담을 줄이기 위해 **네 개의 층** 중 DB가 아는 것은 자동으로 뽑고, 사람만 아는 뜻만 `semantic_layer.yaml`에서 읽어 합칩니다.

| 층 (Layer) | 담는 내용 | 누가 만드나 | 실제 예시 (`enterprise_erp`) | 없으면 생기는 문제 |
|:---|:---|:---:|:---|:---|
| **① 원시 스키마** | 컬럼명, 타입, PK, FK(조인 경로), `CHECK` 허용값, 샘플 2행 | **DB 자동 추출**<br/>(`PRAGMA`) | `approval_status` ∈ `{'승인완료', '결재대기', '반려'}` | 없는 컬럼을 지어내거나 조인 키를 틀림 |
| **② 데이터 사전** | 테이블·컬럼의 업무 설명과 현업 동의어 | 데이터 담당자<br/>(`YAML`) | `expense_transactions` 동의어 = `품의, 전표, 경비` | `"품의 내역"`이라고 물으면 테이블을 못 찾음 |
| **③ 업무 규칙** | 모호한 용어의 기본 해석(`global_rules`, `rules`) | 업무 담당자<br/>(`YAML`) | *"실집행액은 승인완료 건만 합산"*, *"예산은 프로젝트/부서 2종류"* | 결재대기·반려까지 합산해 숫자를 부풀림 |
| **④ 지표 (Metrics)** | 컬럼으로 없지만 자주 묻는 계산식·단위·조인 힌트 | 업무 담당자<br/>(`YAML`) | `budget_burn_rate` = `승인완료 합계 * 100.0 / 승인 예산` | 질문할 때마다 다른 수식으로 계산함 |

> 💡 **설계 원칙 — DB 밖의 규정 기준값은 시멘틱 레이어에 적지 않습니다!**  
> 직급별 출장 숙박비 한도(제14조)나 위임전결 금액 기준처럼 **사내 규정 문서에 정의된 기준값**은 개정 주기와 관리 주체가 DB와 다릅니다. 이런 값은 YAML에 복사해 두지 않고, 런타임에 에이전트가 규정 검색 도구로 찾아 `context`(참고 기준값)로 넘기도록 분리합니다(Part 6에서 실습).

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    DB[("enterprise_erp.db<br/>① PRAGMA 자동 추출<br/>(컬럼·PK/FK·허용값·샘플행)")] --> PROF["DatabaseProfile.load()"]
    YAML["semantic_layer.yaml<br/>② 설명·동의어<br/>③ 업무 규칙 · ④ 지표 공식"] --> PROF
    PROF --> OV["profile.overview()<br/>DB 마스터 요약 + 공통 규칙<br/>(짧아서 프롬프트에 항상 주입)"]
    PROF --> TC["profile.table_card(name)<br/>테이블 상세 카드 7장<br/>(질문에 필요한 카드만 검색 주입)"]
    PROF --> MC["profile.metric_card(key)<br/>지표 공식 카드 6장<br/>(질문에 필요한 카드만 검색 주입)"]

    style DB fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style YAML fill:#fff8e1,stroke:#f57f17,stroke-width:1.5px
    style PROF fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
```

### 2.1 `DatabaseProfile` 로드 및 마스터 개요(`overview`) 확인

`DatabaseProfile.load()`를 호출해 물리 DB와 `semantic_layer.yaml`이 합쳐진 결과를 확인해 봅시다. `overview()`는 전체 테이블 한 줄 요약과 전사 공통 규칙(`global_rules`)을 담고 있어 모든 SQL 생성 프롬프트 상단에 기본 주입됩니다.

In [ ]:
from rag.sql import DatabaseProfile

profile = DatabaseProfile.load(SEMANTIC_LAYER_PATH, db_path=ERP_DB_PATH)

print(f"✅ 로드 완료: {profile.title} ({profile.dialect})")
print(f"  - 테이블 수 : {len(profile.tables)}개 -> {list(profile.tables.keys())}")
print(f"  - 지표 수   : {len(profile.metrics)}개 -> {list(profile.metrics.keys())}")
print("\n" + "=" * 75)
print(profile.overview())

### 2.2 Text2SQL의 '청크(Chunk)': 테이블 카드(`table_card`)와 지표 카드(`metric_card`)

문서 RAG에서 긴 규정을 **청크(Chunk)** 단위로 잘라 색인하듯, Text2SQL에서는 **테이블 1개당 카드 1장(`table_card`), 지표 1개당 카드 1장(`metric_card`)** 으로 묶어 검색 단위로 사용합니다.  
데이터 행 수천 건을 벡터화하는 대신, **구조와 업무 뜻이 응축된 이 카드들만 색인**하는 것이 핵심입니다. `expense_transactions` 테이블 카드와 `budget_burn_rate` 지표 카드를 직접 출력해 봅시다.

In [ ]:
print("🃏 [테이블 카드 예시: expense_transactions]")
print(profile.table_card("expense_transactions"))

print("\n" + "=" * 75)
print("📐 [지표 카드 예시: budget_burn_rate]")
print(profile.metric_card("budget_burn_rate"))

출력된 카드를 보면 DB에서 자동 추출한 `[허용값: '승인완료', '결재대기', '반려']`, `조인 경로(FK)`, `샘플 행` 위에 사람이 작성한 `동의어: 지출, 지출결의, 품의, 전표...`, `규칙`, `SQL 식`이 한 장의 깔끔한 문서로 합쳐져 있습니다.

### 2.3 운영 안전장치: 스키마 변경 자동 감지 (Schema Drift Guard)

실무에서는 DBA가 DB 컬럼명을 바꿨는데 시멘틱 레이어(`yaml`) 수정을 깜빡해 장애가 나는 경우가 잦습니다.  
`_apply_table_spec()`은 로드 시점에 실제 DB에 없는 테이블이나 컬럼이 YAML에 적혀 있으면 즉시 `ValueError`를 발생시켜 배포 전에 불일치를 잡아냅니다.

In [ ]:
from rag.sql.semantic_layer import _apply_table_spec

try:
    # 실제 projects 테이블에는 없는 'spent_budget_krw' 컬럼을 시멘틱 레이어에 잘못 적은 상황 시뮬레이션
    broken_spec = {
        "projects": {
            "columns": {
                "spent_budget_krw": "실집행액 (DB에는 존재하지 않는 컬럼)"
            }
        }
    }
    _apply_table_spec(profile.tables, broken_spec)
except ValueError as e:
    print(f"🛡️ 스키마 불일치 즉시 감지 성공! -> ValueError: {e}")

---
## 🧭 Part 3. 스키마 링킹 (Schema Linking): 필요한 장부와 조인 경로만 골라내기

### 스키마 링킹이란 무엇인가요?

현업 ERP·데이터웨어하우스에는 수백 개의 테이블이 존재합니다. 모든 테이블의 상세 카드를 프롬프트에 다 넣으면 컨텍스트가 넘칠 뿐 아니라, 이름이 비슷한 컬럼(`projects.status` vs `expense_transactions.approval_status`, `projects` 예산 vs `department_budgets` 예산)이 뒤섞여 모델이 엉뚱한 테이블을 `JOIN`합니다.

**스키마 링킹(Schema Linking)** 은 자연어 질문 속 단어를 분석해 **이번 SQL의 `FROM` · `JOIN` · `SELECT`에 꼭 필요한 테이블과 지표만 연결하는 과정**입니다.  
주목할 점은 `3_Vector_and_Graph_RAG.ipynb`에서 문서 청크를 찾을 때 썼던 **`HybridRetriever`와 100% 동일한 4단계 아키텍처**를 스키마 카드에 그대로 적용한다는 것입니다!

| 단계 | 문서 RAG (`3_Vector_and_Graph_RAG` · `HybridRetriever`) | 스키마 RAG (`4_Text2SQL` · `SchemaRetriever`) |
|:---:|:---|:---|
| **검색 단위** | 정규화된 규정·백서 청크 (`Document`) | **테이블 카드(7장) + 지표 카드(6장) = 총 13장 (`Document`)** |
| **1단계** | `DomainDictionary` 사내 약어 정규화 | **동일** (`"출장비"` → `"국내 여비 및 교통비"`, `"클운팀"` → `"클라우드운영팀"`) |
| **2단계** | `BM25` + `Dense` 병렬 검색 | **동일** (단, BM25 점수가 `0`인 무관 카드는 제외) |
| **3단계** | `RRF` 순위 융합 | **동일** + 지표 카드가 뽑히면 그 지표의 `필요 테이블`을 자동 추가 |
| **4단계** | `ContextualWindow` (전후 인접 청크 확장) | **상위 방향 FK 확장 (`expand`, `fk_hops=1`)** — 참조하는 부모 마스터 테이블 추가 |

### 왜 FK 확장은 '상위(부모) 방향'으로만 따라갈까요?

외래키(FK)에는 방향이 있습니다. 예를 들어 `expense_transactions`(전표)는 `requester_emp_id → employees.emp_id`로 임직원 테이블을 가리킵니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart BT
    TX["expense_transactions<br/>(지출 전표)"] -->|"FK: requester_emp_id<br/>✅ 상위(부모) 확장 O"| EMP["employees<br/>(임직원 마스터)"]
    TX -->|"FK: project_id<br/>✅ 상위(부모) 확장 O"| PRJ["projects<br/>(전략 프로젝트)"]
    PM["project_members"] -.->|"❌ 하위 확장 금지"| EMP
    DEPT["departments"] -.->|"❌ 하위 확장 금지"| EMP

    style TX fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px
    style EMP fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style PRJ fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

- **상위(부모) 방향 확장 (`✅ 필수`)**: 지출 전표(`expense_transactions`)만 검색되었을 때 기안자의 **이름·직급**이나 **프로젝트명**을 함께 보여주려면 부모 테이블인 `employees`, `projects`와 반드시 `JOIN`해야 합니다.
- **하위(자식) 방향 확장 (`❌ 금지`)**: 반대로 *"수석 직급 직원 명단 알려줘"* 처럼 `employees`가 검색되었을 때 `employees`를 참조하는 하위 테이블을 모두 따라가면 프로젝트, 투입인력, 지출전표, 부서까지 **거의 모든 DB 테이블이 딸려 오기 때문**입니다.

### 3.1 `SchemaRetriever` 검색 및 상위 FK 확장 실험

질문 의도에 따라 **프로젝트 예산**과 **부서 운영 예산**이 어떻게 구분되어 검색되는지, 그리고 `expand()`가 부모 테이블을 어떻게 자동으로 챙겨 주는지 확인해 봅시다.

In [ ]:
from rag.sql import SchemaRetriever

schema_retriever = SchemaRetriever(profile, embeddings=embeddings, fk_hops=1)
print(f"🗂️ 색인된 스키마 카드 총 {len(schema_retriever.cards)}장 (테이블 {len(profile.tables)} + 지표 {len(profile.metrics)})\n")

test_queries = [
    "진행 중인 프로젝트 중 집행률이 80%를 넘는 과제를 알려줘",
    "부서별 운영비 예산 편성액과 집행 현황을 보여줘",
    "승인 대기 중인 출장 품의 내역을 보여줘",
]

for q in test_queries:
    res = schema_retriever.search(q, top_k=3)
    print(f"🔎 질문: {q}")
    print(f"  1) 약어 정규화 : {res.normalized_query}")
    print(f"  2) 직접 검색됨 : {res.retrieved}")
    print(f"  3) 선택된 지표 : {res.metrics}")
    print(f"  4) 최종 테이블 (FK 1-hop 확장 포함): {res.tables}")
    print("-" * 75)

### 3.2 스키마 링킹만으로는 해결되지 않는 두 가지 빈틈!

위 결과를 자세히 보면 스키마 검색만으로는 아직 완벽한 SQL을 짤 수 없는 두 가지 순간이 보입니다.

1. **2단계(2-hop) 너머에 있는 조건 테이블 누락**:
   - `expense_transactions`에서 1-hop 부모를 펼치면 `['expense_transactions', 'projects', 'employees']`까지만 옵니다(`expense_transactions → employees → departments`는 **2-hop**).
   - 만약 질문이 `"클운팀의 결재대기 출장 품의"`처럼 `'부서'`라는 단어 없이 **값(`'클운팀'`)** 만 말한다면, `departments` 테이블이 후보에서 빠질 위험이 있습니다!
2. **컬럼 안에 실제로 저장된 글자(Literal)를 모름**:
   - 설령 `departments`나 `employees` 테이블 카드를 받았더라도, 샘플 행은 딱 2줄뿐입니다. 질문 속 `'샌프란'`, `'정수석'`, `'이선임'`이 **어느 테이블의 어느 컬럼에 정확히 어떤 문자열로 저장되어 있는지**는 카드 검색만으로는 알 수 없습니다.

이 빈틈을 메워 주는 단짝 단계가 바로 **Part 4의 값 연결(Value Linking)** 입니다!

In [ ]:
# 확인: expense_transactions 단일 테이블에서 fk_hops=1 확장을 수행하면?
one_hop_tables = schema_retriever.expand(["expense_transactions"])
print("📌 expense_transactions의 1-hop 상위 FK 확장 결과:", one_hop_tables)
print("   👉 기안자의 소속 부서명(departments.dept_name)까지 가려면 2-hop이므로 'departments'가 아직 없습니다!")

---
## 🔗 Part 4. 값 연결 (Value Linking): 일상 표현을 DB 실제 셀 값에 글자 그대로 매칭하기

### 값 연결(Value Linking)이란 무엇인가요?

벡터 검색은 `'클운팀'`과 `'클라우드운영팀'`의 의미가 비슷하다고 찾아주지만, **SQL의 `WHERE dept_name = '클운팀'`은 글자가 한 글자라도 다르면 가차 없이 `0행(FALSE)`을 반환**합니다.  
**값 연결(Value Linking)** 은 질문 속에 등장하는 고유명사·상태값·약어를 미리 색인해 둔 **DB 실제 저장 값(Cell Value)** 과 대조하여, *"질문의 이 표현은 `테이블.컬럼 = '실제값'`으로 쓰세요"* 라고 콕 집어 알려주는 단계입니다.

### `ValueLinker`의 3가지 핵심 설계 규칙 (`rag/sql/value_linker.py`)

1. **선택적 값 색인 (Selective Indexing — 노이즈 차단)**:
   - 금액(`150000`), 날짜(`2026-03-12`), 긴 적요(`description`)까지 다 색인하면 엉뚱한 매칭만 늘어납니다.
   - 오직 **① DDL `CHECK` 제약 허용값**(`'승인완료'`, `'결재대기'`, `'수석'` 등 자동 추출)과 **② `semantic_layer.yaml`에서 `value_lookup: true`로 지정한 식별자·명칭 컬럼**(`dept_name`, `employees.name`, `project_name`, `destination_city` 등)의 `SELECT DISTINCT` 값만 깔끔하게 색인합니다.
2. **약어 정규화 후 '최장 일치 우선(Longest-Span Match)' 적용**:
   - 먼저 `DomainDictionary`로 약어(`'클운팀'` → `'클라우드운영팀'`, `'샌프란'` → `'샌프란시스코'`)를 표준 명칭으로 바꿉니다.
   - 이어서 한국어 사내 데이터의 대표 함정을 막기 위해 **긴 문자열이 차지한 구간 안에 완전히 포함되는 짧은 문자열은 버립니다.**

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart TD
    Q["질문: '정수석이 총괄 PM인 과제와 수석 직급 인원수를 알려줘'"] --> S1["구간 [0:3] '정수석' (길이 3)<br/>→ employees.name ✅ 채택!"]
    Q --> S2["구간 [1:3] '수석' (길이 2)<br/>→ '정수석' 구간 내부에 포함됨 ❌ 폐기!"]
    Q --> S3["구간 [16:18] '수석' (길이 2)<br/>→ 독립된 구간 ✅ employees.rank 채택!"]

    style S1 fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style S2 fill:#ffebee,stroke:#c62828,stroke-width:1.5px
    style S3 fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

3. **동일 값 다중 컬럼 안내 & 누락 테이블 스키마 역보강 (Schema Backfill)**:
   - `'국내출장'`처럼 `expense_transactions.category`와 `department_budgets.category` 두 곳에 모두 존재하는 값은 **두 컬럼 후보를 모두 출력**해 SQL 생성 모델이 질문 맥락에 맞춰 선택하게 합니다.
   - 만약 값 연결에서 `'클라우드운영팀' → departments.dept_name`이 발견되었는데 앞선 스키마 검색 결과에 `departments`가 빠져 있었다면, **누락된 `departments` 테이블(및 그 부모 FK)을 스키마 컨텍스트에 자동으로 보강**합니다!

### 4.1 `ValueLinker` 매칭 실험 및 스키마 역보강 확인

In [ ]:
from rag.sql import ValueLinker

value_linker = ValueLinker(profile)
print(f"📇 ValueLinker 색인 완료: 총 {len(value_linker.index)}개의 고유 셀 값 색인됨")

sample_keys = ["클라우드운영팀", "정수석", "수석", "국내출장", "샌프란시스코"]
for k in sample_keys:
    print(f"  • '{k}' -> {value_linker.index.get(k)}")

print("\n" + "=" * 75)
vl_questions = [
    "클운팀의 결재대기 국내출장 품의 금액 합계는?",
    "정수석이 총괄 PM인 과제와 수석 직급 인원수를 알려줘",
    "샌프란 출장 전표 내역 보여줘",
]

for q in vl_questions:
    matches = value_linker.link(q)
    print(f"❓ 질문: {q}")
    print(ValueLinker.render(matches))
    print("-" * 75)

In [ ]:
# 스키마 링킹(Part 3) + 값 연결(Part 4) 상호 보완 시연:
# expense_transactions만 선택되어 'departments'가 빠진 상태에서 "클운팀 결재대기 국내출장" 값 연결이 일어나면?
selected_tables = schema_retriever.expand(["expense_transactions"])  # ['expense_transactions', 'projects', 'employees']
matches = value_linker.link("클운팀의 결재대기 국내출장 품의 금액 합계는?")

value_tables = [col.split(".")[0] for m in matches for col in m.columns]
missing_tables = [t for t in dict.fromkeys(value_tables) if t not in selected_tables]

print(f"1️⃣ 스키마 검색 직후 테이블 : {selected_tables}")
print(f"2️⃣ 값 연결이 찾아낸 테이블 : {list(dict.fromkeys(value_tables))}")
print(f"3️⃣ 누락 감지되어 보강될 테이블 : {missing_tables}")

if missing_tables:
    selected_tables += [t for t in schema_retriever.expand(missing_tables) if t not in selected_tables]
print(f"4️⃣ 보강 완료된 최종 테이블 : {selected_tables}")

---
## 🛡️ Part 5. 읽기 전용 2중 가드레일과 LangGraph Self-Healing 워크플로우

재료를 완벽히 준비했더라도 **LLM이 생성한 SQL을 그대로 운영 DB에 실행하는 것은 금물**입니다.  
사내 정보보안관리지침 및 프로덕션 표준에 따라 `rag/sql/executor.py`와 `rag/sql/pipeline.py`는 **2중 안전 가드레일**과 **실패를 피드백으로 삼는 자가 수정(Self-Healing) 루프**를 결합합니다.

### 5.1 읽기 전용 2중 가드레일 (`rag/sql/executor.py`)

| 방어 계층 | 구현 메커니즘 | 차단하는 위험 |
|:---|:---|:---|
| **1단계: 코드 레벨 정규식 검수**<br/>(`validate_sql`) | • 문자열 리터럴(`'...삭제...'`)과 주석을 먼저 발라내어 **오탐(False Positive) 방지**<br/>• 단일 `SELECT` / `WITH` 시작 강제, `;` 다중 쿼리 차단<br/>• `INSERT`, `UPDATE`, `DELETE`, `DROP`, `PRAGMA`, `ATTACH` 등 금지 | 데이터 위변조·삭제, SQL Injection 다중 명령 실행 |
| **2단계: DB 엔진 권한 & 리소스 제한**<br/>(`connect_readonly`, `run_readonly_query`) | • SQLite URI `?mode=ro`(Read-Only 파일 핸들) + `PRAGMA query_only = ON`<br/>• `set_progress_handler` 기반 **실행 타임아웃(5초)** 강제 중단<br/>• 최대 **50행(`max_rows`)** 까지만 반환하되 전체 행 수를 함께 집계 | 정규식 우회 쓰기 시도, 카테시안 곱(잘못된 JOIN)으로 인한 DB 먹통 및 컨텍스트 폭발 |

특히 실행 중 SQL 오류가 나더라도 파이썬 예외(`Exception`)로 죽지 않고 `ExecutionResult(ok=False, error=...)` 객체에 **DB 오류 메시지를 담아 반환**하므로, 다음 단계에서 LLM이 이 메시지를 읽고 스스로 쿼리를 고칠 수 있습니다.

In [ ]:
from rag.sql import validate_sql, run_readonly_query

guardrail_cases = [
    ("쓰기 명령 차단 (DELETE)", "DELETE FROM expense_transactions WHERE tx_id = 'TX-2026-0001'"),
    ("다중 문장 차단 (; DROP)", "SELECT project_id FROM projects; DROP TABLE projects"),
    ("문자열 내부 키워드 허용 (오탐 방지)", "SELECT tx_id, description FROM expense_transactions WHERE description LIKE '%삭제%'"),
    ("없는 컬럼 조회 (Self-Healing용 에러 반환)", "SELECT project_id, spent_budget_krw FROM projects"),
]

for title, sql in guardrail_cases:
    cleaned, v_err = validate_sql(sql)
    exec_res = run_readonly_query(ERP_DB_PATH, sql)
    print(f"🧪 [{title}]")
    print(f"  • SQL        : {sql}")
    print(f"  • 1차 검증   : {'통과 (None)' if v_err is None else f'차단 -> {v_err}'}")
    print(f"  • 실행 결과  : ok={exec_res.ok} | {exec_res.summary() if exec_res.ok else exec_res.error}")
    print("-" * 75)

### 5.2 `Text2SQL` LangGraph 워크플로우 조립 (`rag/sql/pipeline.py`)

이제 지금까지 살펴본 모든 부품을 하나의 LangGraph 상태 그래프(`Text2SQL`)로 조립합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    ST(["START"]) --> RS["1. retrieve_schema<br/>카드 검색 + 상위 FK 확장"]
    RS --> LV["2. link_values<br/>값 매칭 + 누락 테이블 보강"]
    LV --> GS["3. generate_sql<br/>구조화 출력 (SQLDraft)<br/>sql · assumptions · needs_clarification"]
    GS -->|"needs_clarification = True<br/>(해석 후보와 함께 되묻기)"| ED(["END<br/>Text2SQLResult"])
    GS -->|"단일 SQL 생성됨"| EX["4. execute_sql<br/>읽기 전용 2중 가드레일 실행"]
    EX -->|"SQL 오류 OR 첫 0행(empty_checked=False)<br/>[이전 시도 피드백 주입 · 최대 3회]"| GS
    EX -->|"실행 성공 (ok)"| ED

    style RS fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style LV fill:#fff3e0,stroke:#fb8c00,stroke-width:1.5px
    style GS fill:#ede7f6,stroke:#5e35b1,stroke-width:1.5px
    style EX fill:#ffebee,stroke:#d32f2f,stroke-width:1.5px
    style ED fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

이 그래프에는 실무 신뢰성을 결정짓는 **두 가지 조건부 분기**가 들어 있습니다.
1. **모호성 방어 분기 (`_after_generate`)**: 시멘틱 레이어의 기본 규칙으로도 해석이 하나로 정해지지 않으면(예: *"예산 집행률이 80% 이상인 항목"* — 프로젝트 예산 vs 부서 운영 예산), 임의로 찍어서 틀린 표를 내는 대신 `needs_clarification=True`와 선택지(`candidates`)를 담아 즉시 반환합니다. 기본 규칙을 적용해 해석했을 때는 `assumptions`(적용한 가정)에 근거를 투명하게 남깁니다.
2. **자가 수정 루프 (`_after_execute`, Self-Healing)**:
   - **SQL 실행 오류 시**: SQLite가 돌려준 오류(`no such column` 등)를 `[이전 시도 피드백]`에 넣어 SQL을 다시 작성하게 합니다.
   - **결과 0행일 때 1회 재검증 (`empty_checked`)**: 결과가 `0행`이면 문법 오류는 아니지만 `WHERE` 조건 값을 잘못 적었을 가능성이 있으므로, *"DB 실제 값과 글자 그대로 일치하는지 한 번 더 점검하고, 정말 0행이 맞다면 같은 SQL을 반환하라"* 고 한 번 되짚어 조용한 오답을 예방합니다.

### 5.3 Part 1 실패 질문 재도전! (Naive Text2SQL vs 시멘틱 레이어 `Text2SQL`)

Part 1에서 `0행`과 `조용한 오답`을 냈던 두 질문, 그리고 구체화된 **시나리오 A(집행률 80% 초과 전략 프로젝트 조기경보)** 를 `Text2SQL`에 그대로 던져 결과를 비교해 봅시다.

In [ ]:
from rag.sql import Text2SQL

erp_t2s = Text2SQL.from_semantic_layer(
    SEMANTIC_LAYER_PATH,
    llm=llm,
    db_path=ERP_DB_PATH,
    embeddings=embeddings,
)

# 1. Part 1에서 0행이 나왔던 질문 재도전 ('클라우드팀' -> '클라우드운영팀' 매핑 및 가정 명시)
res_cloud = erp_t2s.invoke("클라우드팀의 연간 운영 예산과 비목별 편성액을 알려줘.")
print("🎯 [재도전 1: '클라우드팀' 예산 조회]")
print(res_cloud.to_text())

print("\n" + "=" * 75)

# 2. Part 1에서 한쪽 테이블만 멋대로 조회했던 모호한 질문 재도전 -> 되묻기(needs_clarification) 작동!
res_ambig = erp_t2s.invoke("예산 집행률이 80% 이상인 항목을 모두 알려줘.")
print("🎯 [재도전 2: 모호한 '예산 집행률 80% 이상' 질문 -> 조용한 오답 대신 되묻기!]")
print(res_ambig.to_text())

print("\n" + "=" * 75)

# 3. 사용자가 후보 중 '전사 전략 프로젝트'를 선택해 구체화한 경우 (시나리오 A)
res_scen_a = erp_t2s.invoke(
    "현재 진행 중인 전사 전략 프로젝트 중 예산 집행률이 80%를 초과하는 프로젝트의 "
    "코드, 명칭, 주관 부서, 총괄 PM, 승인 예산, 실집행액, 집행률(%)을 구해줘."
)
print("🎯 [시나리오 A: 전사 전략 프로젝트 집행률 80% 초과 조기경보 검출]")
print(res_scen_a.to_text())

### 5.4 그래프 내부 추적: 오류 복구 및 0행 재검증(Self-Healing) 루프 눈으로 확인하기

`Text2SQL` 내부에서 실제로 피드백 루프가 어떻게 도는지 확인해 보겠습니다.
1. **0행 재검증 루프 추적**: 실제로 조건에 해당하는 데이터가 없는 질문(*"진행 중인 전략 프로젝트 중 집행률이 95%를 초과하는 과제"*)을 던지면 `generate_sql` ➔ `execute_sql`(0행 감지!) ➔ `generate_sql`(조건 재점검) ➔ `execute_sql` 순서로 **총 2회(`attempts=2`)** 신중하게 검증한 뒤 최종 `0행`을 확정합니다.
2. **컬럼 오타 자가 수정 확인**: 만약 1차 시도에서 모델이 실수로 없는 컬럼(`spent_budget_krw`)을 조회해 SQLite 에러 피드백이 들어왔을 때, `_generate_sql` 노드가 피드백을 읽고 어떻게 올바른 `JOIN` 집계 쿼리로 고쳐 내는지 확인합니다.

In [ ]:
# 1) 0행 발생 시 자동 재검증(empty_checked) 그래프 스트리밍 추적
zero_row_q = "현재 진행 중인 전사 전략 프로젝트 중 예산 집행률이 95%를 초과하는 프로젝트를 알려줘."
print(f"🔄 [스트리밍 추적] 질문: {zero_row_q}\n")

for step_num, event in enumerate(erp_t2s.graph.stream({"question": zero_row_q, "context": "", "attempts": 0}), 1):
    node_name, node_out = next(iter(event.items()))
    if node_name == "retrieve_schema":
        print(f"  Step {step_num} [{node_name}] 선택 테이블: {node_out['selected_tables']} | 지표: {node_out['selected_metrics']}")
    elif node_name == "link_values":
        print(f"  Step {step_num} [{node_name}] 값 힌트:\n    " + node_out["value_hints"].replace("\n", "\n    "))
    elif node_name == "generate_sql":
        print(f"  Step {step_num} [{node_name}] (시도 {node_out['attempts']}회차) SQL: {node_out['draft'].sql}")
    elif node_name == "execute_sql":
        r = node_out["result"]
        fb_msg = "재검증 피드백 발생! -> generate_sql로 루프" if node_out.get("feedback") else "검증 완료 -> 종료"
        print(f"  Step {step_num} [{node_name}] 결과: {r.summary()} | {fb_msg}")

print("\n" + "=" * 75)

# 2) SQL 오류 피드백을 통한 쿼리 자가 수정(Self-Correction) 시뮬레이션
q_fix = "프로젝트별 승인 예산과 실집행액을 조회해줘."
schema_res = erp_t2s.schema_retriever.search(q_fix, top_k=3)
error_state = {
    "question": q_fix,
    "context": "",
    "schema_context": schema_res.context,
    "value_hints": ValueLinker.render(erp_t2s.value_linker.link(q_fix)),
    "attempts": 1,
    "feedback": (
        "이전 SQL:\nSELECT project_id, project_name, approved_budget_krw, spent_budget_krw FROM projects\n"
        "문제: SQLite 오류: no such column: spent_budget_krw\n오류를 고친 SQL을 다시 작성하세요."
    ),
}
healed_out = erp_t2s._generate_sql(error_state)
healed_exec = run_readonly_query(ERP_DB_PATH, healed_out["draft"].sql)
print("🩹 [SQL 오류 피드백 기반 자가 수정 결과]")
print(f"  • 주입된 오류 피드백 : SQLite 오류: no such column: spent_budget_krw")
print(f"  • 수정된 SQL (2회차) : {healed_out['draft'].sql}")
print(f"  • 재실행 결과 ({healed_exec.summary()}) :\n{healed_exec.to_markdown()}")

---
## 🤝 Part 6. 에이전트 도구로 결합하기: 비정형 규정 × 정형 ERP 하이브리드 감사

실무에서 가장 강력한 Agentic RAG 패턴은 **비정형 문서(사내 규정집)** 와 **정형 DB(ERP 전표)** 를 넘나들며 추론하는 것입니다.  
`app/agents/tool_rag_agent/tools.py`에 정의된 `search_enterprise_erp` 도구의 시그니처를 살펴보면 왜 인자가 `query`와 `context` 두 개로 설계되어 있는지 알 수 있습니다.

```python
@tool
def search_enterprise_erp(query: str, context: str = "") -> str:
    """사내 ERP 정형 데이터베이스에서 실시간 수치와 현황 레코드를 조회합니다.
    query에는 SQL 문이 아니라 자연어 질문을 넣으세요. SQL은 이 도구가 스키마를 보고 직접 생성합니다."""
    return _erp_text2sql().invoke(query, context=context).to_text()
```

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    U["사용자 감사 질의<br/>(여비규정 위반 전표 적발)"] --> AG["메인 ReAct 에이전트<br/>(DB 스키마를 전혀 모름)"]
    AG -->|"① 규정 조회"| POL["search_company_policy<br/>(Vector + BM25)"]
    POL -->|"제14조 직급·지역별<br/>1박 숙박비 상한표"| AG
    AG -->|"② query + context(제14조 한도)"| ERP["search_enterprise_erp<br/>(Text2SQL 워크플로우)"]
    ERP -->|"CASE WHEN으로 한도 반영 후<br/>위반 전표 3건 반환"| AG
    AG --> ANS["최종 감사 리포트 작성"]

    style AG fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px
    style POL fill:#f3e5f5,stroke:#8e24aa,stroke-width:1.5px
    style ERP fill:#fff3e0,stroke:#fb8c00,stroke-width:1.5px
    style ANS fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

- **메인 에이전트의 컨텍스트 격리**: 메인 에이전트는 7개 테이블의 컬럼명이나 SQL 문법을 전혀 알 필요가 없습니다. DB가 10개로 늘어나도 메인 에이전트에는 가벼운 도구 설명만 추가될 뿐 프롬프트가 비대해지지 않습니다.
- **`context` 인자를 통한 외부 기준값 동적 주입**: ERP DB 안에는 직급·지역별 숙박비 한도 금액이 없습니다. 메인 에이전트가 먼저 `search_company_policy`로 **제14조 숙박비 상한표**를 읽은 뒤, 이를 `search_enterprise_erp(query=..., context="...")`로 넘기면 `Text2SQL`이 `CASE WHEN` 구문으로 SQL에 즉시 녹여냅니다! 사내 규정이 개정되어도 Text2SQL 코드는 한 줄도 수정할 필요가 없습니다.

### 6.1 실습: 여비 규정(제14조) 위반 '결재대기' 국내출장 전표 적발 (시나리오 B)

`search_company_policy`와 `search_enterprise_erp` 두 도구를 장착한 감사 에이전트를 만들고, 비정형 규정 검색과 정형 ERP SQL 감사가 자동으로 이어지는 과정을 실행해 봅시다.

In [ ]:
from langchain.agents import create_agent
from app.agents.tool_rag_agent.tools import search_company_policy, search_enterprise_erp

AUDIT_SYSTEM_PROMPT = """당신은 (주)넥스트AI의 내부감사 AI 에이전트입니다.
1. 규정 기준(한도 금액, 전결 기준 등)이 필요한 질문은 반드시 먼저 `search_company_policy`로 정확한 조항과 수치를 확인하세요.
2. 확인한 규정 기준값을 `search_enterprise_erp`의 `context` 인자에 구체적으로 전달하여 ERP 데이터를 조회하세요.
3. 조회된 전표 번호, 기안자 성명·직급·부서, 출장지·박수, 청구 금액, 규정 상한액, 초과 금액을 표로 명확히 정리해 보고하세요."""

audit_agent = create_agent(
    model=llm,
    tools=[search_company_policy, search_enterprise_erp],
    system_prompt=AUDIT_SYSTEM_PROMPT,
)

scenario_b_question = (
    "사내 국내 여비 및 교통비 지급 규정(제14조)의 직급별·지역별 숙박비 상한액을 기준으로, "
    "현재 ERP에서 '결재대기' 중인 '국내출장' 품의 가운데 1박당 숙박비 한도를 초과하여 위반한 내역을 모두 찾아 보고해줘."
)

response = audit_agent.invoke({"messages": [{"role": "user", "content": scenario_b_question}]})

print("🔍 [에이전트 도구 호출 궤적 확인]")
for msg in response["messages"]:
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            print(f"  🛠️ 도구 호출: {tc['name']}")
            for k, v in tc["args"].items():
                preview = v if len(str(v)) <= 120 else str(v)[:120] + "..."
                print(f"     - {k}: {preview}")
    elif msg.type == "tool":
        first_line = str(msg.content).splitlines()[0] if msg.content else ""
        print(f"  📥 도구 응답 ({msg.name}): {first_line} (총 {len(str(msg.content)):,}자)")

print("\n" + "=" * 75)
print("📋 [최종 하이브리드 감사 리포트]\n")
print(normalize_content(response["messages"][-1].content))

---
## 🏁 Part 7. 정리 및 다음 단계

이번 노트북에서는 정형 데이터베이스를 Agentic RAG의 신뢰할 수 있는 검색 도구로 편입시키기 위한 **엔터프라이즈 Text2SQL 아키텍처**를 완성했습니다.

### 핵심 설계 원칙 요약

| 단계 / 원칙 | 핵심 질문 | 구현 모듈 및 해법 |
|:---|:---|:---|
| **1. 시멘틱 레이어**<br/>*(Semantic Layer)* | *"컬럼 이름만 있는 DB에 현업의 뜻과 공식을 어떻게 입힐까?"* | `DatabaseProfile`: DB 자동 추출(`PRAGMA`) + `semantic_layer.yaml`(설명·규칙·지표 공식) 결합 → **테이블·지표 카드** 생성 |
| **2. 스키마 링킹**<br/>*(Schema Linking)* | *"수십 개 테이블 중 질문에 필요한 테이블과 JOIN 경로만 어떻게 고를까?"* | `SchemaRetriever`: 약어 정규화 → `BM25 + Dense` → `RRF` → **상위(Parent) 방향 1-hop FK 확장** |
| **3. 값 연결**<br/>*(Value Linking)* | *"사용자의 일상 표현·약어가 `WHERE` 절에서 `0행`이 되지 않게 하려면?"* | `ValueLinker`: 허용값·식별자 선택 색인 + **최장 일치 매칭**(`'정수석'` vs `'수석'`) + **누락 테이블 역보강** |
| **4. 안전 실행 & 자가 수정**<br/>*(Guardrail & Self-Healing)* | *"위험한 쿼리와 조용한 오답을 어떻게 막을까?"* | `executor.py` + `Text2SQL`: 정규식·`mode=ro` 2중 가드레일, 오류·0행 피드백 재생성, 모호한 질문 **되묻기(`needs_clarification`)** |
| **5. 도구 경계 캡슐화**<br/>*(Tool Encapsulation)* | *"비정형 규정과 정형 DB를 어떻게 깔끔하게 결합할까?"* | `search_enterprise_erp(query, context)`: SQL과 스키마는 도구 내부에 숨기고, 외부 규정 기준값만 `context`로 주입 |

---

### 💪 스스로 해보기 (심화 연습)

위 `audit_agent`에 다음 질문(**시나리오 E: 부서 운영 예산 통제 등급 감사**)을 던져 보세요!  
이번에는 프로젝트 예산(`projects`)이 아니라 **부서 운영 예산(`department_budgets`)** 테이블과 **조직장 자기참조 조인(`employees.manager_emp_id`)** 이 자동으로 선택되는지 확인해 볼 수 있습니다.

```python
# 연습용 질문 예시 (새 코드 셀을 만들어 실행해 보세요)
practice_q = (
    "사내 전사위임전결및예산집행규정(제11조)의 집행률 단계별 통제 기준을 먼저 확인한 뒤, "
    "부서 운영 예산 중 집행률이 80% 이상인 비목을 모두 찾아 통제 등급(조기 경보/동결 경계/차단)과 "
    "경보를 받아야 할 부서장 및 소속 본부장을 보고해줘."
)
```

---

### 👉 다음 단계: `5_MCP_and_Skills.ipynb`

지금까지 우리는 사내 규정 Vector DB, AWS 백서 DB, 지식 그래프(`3_Vector_and_Graph_RAG.ipynb`), 그리고 사내 ERP Text2SQL(`4_Text2SQL.ipynb`)까지 **4대 엔터프라이즈 데이터 검색기**를 모두 구축했습니다.  
다음 **`5_MCP_and_Skills.ipynb`** 에서는 이 검색 도구들을 에이전트 코드에서 분리해 독립된 **MCP(Model Context Protocol) 마이크로서비스 서버**로 격리하고, **Bearer 인증 · 2층 인가(Scope × 보안 등급)** 및 **Skills 기반 점진적 공개** 아키텍처로 확장합니다.